In [ ]:
# ==============================================================================
# MobileNet Crack Classification Model
# ==============================================================================
#
# Binary image classification model for detecting:
#   0 - Mortar cracks
#   1 - Brick cracks
#
# Workflow:
#   1. Load and prepare image dataset
#   2. Apply MobileNet preprocessing and augmentation
#   3. Optimize hyperparameters using Optuna
#   4. Train MobileNet using transfer learning + fine-tuning
#   5. Evaluate model performance
#   6. Export trained model to ONNX format
#
# The original training strategy, model architecture, and hyperparameters
# are preserved.
#
# ==============================================================================

In [ ]:
# ==============================================================================
# Install Required Packages
# ==============================================================================

!pip install optuna
!pip install tf2onnx==1.16.1

In [ ]:
# ==============================================================================
# Imports
# ==============================================================================

import os
import random
import numpy as np

import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

import optuna
import optuna.visualization as vis

from google.colab import drive

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    ReLU,
    GlobalAveragePooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

from tensorflow.keras.applications import MobileNet
from tensorflow.keras.preprocessing.image import (
    load_img,
    img_to_array,
    ImageDataGenerator
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    auc
)

In [ ]:
# ==============================================================================
# Google Drive Setup
# ==============================================================================

drive.mount('/content/drive')


# ==============================================================================
# Dataset Configuration
# ==============================================================================

DATA_DIR = "/content/drive/MyDrive"

MORTAR_DIR = os.path.join(DATA_DIR, "Mortar_Crack")
BRICK_DIR = os.path.join(DATA_DIR, "Brick_Crack")

IMG_SIZE = (224, 224)

In [ ]:
# ==============================================================================
# Dataset Loading Function
# ==============================================================================

def load_images_from_folder(folder_path, label, img_size=IMG_SIZE):
    """
    Load images from a directory and assign the corresponding class label.

    Parameters
    ----------
    folder_path : str
        Directory containing image files.

    label : int
        Binary classification label assigned to all images in the folder.

    img_size : tuple
        Target image dimensions.

    Returns
    -------
    images : list
        Loaded image arrays.

    labels : list
        Corresponding labels.
    """

    images = []
    labels = []

    for filename in os.listdir(folder_path):

        image_path = os.path.join(folder_path, filename)

        image = load_img(
            image_path,
            target_size=img_size
        )

        image_array = img_to_array(image)

        images.append(image_array)
        labels.append(label)

    return images, labels

In [ ]:
# ==============================================================================
# Load Images and Create Dataset
# ==============================================================================

mortar_images, mortar_labels = load_images_from_folder(
    MORTAR_DIR,
    label=0
)

brick_images, brick_labels = load_images_from_folder(
    BRICK_DIR,
    label=1
)


X = np.array(mortar_images + brick_images)
y = np.array(mortar_labels + brick_labels)

In [ ]:
# ==============================================================================
# Train / Validation / Test Split
# ==============================================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=42,
    stratify=y
)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)


print(
    f"Dataset split completed:\n"
    f"Training samples   : {len(X_train)}\n"
    f"Validation samples : {len(X_val)}\n"
    f"Testing samples    : {len(X_test)}"
)

In [ ]:
# ==============================================================================
# Data Augmentation
# ==============================================================================
#
# MobileNet preprocessing is applied before training.
# Augmentation improves model robustness by introducing controlled variations
# in image orientation, position, brightness, and scale.
#
# ==============================================================================

datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet.preprocess_input,

    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,

    shear_range=0.2,
    zoom_range=0.2,

    horizontal_flip=True,
    vertical_flip=True,

    brightness_range=[0.8, 1.2],
    channel_shift_range=0.2,

    fill_mode="reflect"
)

In [ ]:
# ==============================================================================
# Optuna Hyperparameter Optimization
# ==============================================================================

def objective(trial):
    """
    Optuna objective function for MobileNet hyperparameter optimization.

    The objective maximizes validation accuracy by tuning:
        - Learning rate
        - Dropout rate
        - Dense-layer activation function
        - Batch size

    The model architecture and training strategy remain consistent with the
    final training stage.
    """

    # --------------------------------------------------------------------------
    # Hyperparameter Search Space
    # --------------------------------------------------------------------------

    learning_rate = trial.suggest_loguniform(
        "learning_rate",
        1e-5,
        1e-3
    )

    dropout_rate = trial.suggest_float(
        "dropout_rate",
        0.3,
        0.7
    )

    activation = trial.suggest_categorical(
        "activation",
        [
            "relu6",
            "swish",
            "elu"
        ]
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [
            4,
            8,
            16
        ]
    )


    # --------------------------------------------------------------------------
    # Data Generators
    # --------------------------------------------------------------------------

    train_generator = datagen.flow(
        X_train,
        y_train,
        batch_size=batch_size,
        shuffle=True
    )

    validation_generator = datagen.flow(
        X_val,
        y_val,
        batch_size=batch_size,
        shuffle=False
    )


    # --------------------------------------------------------------------------
    # MobileNet Transfer Learning Backbone
    # --------------------------------------------------------------------------

    base_model = MobileNet(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )


    base_model.trainable = True

    # Freeze early layers and allow the final 30 layers to adapt
    for layer in base_model.layers[:-30]:
        layer.trainable = False


    # --------------------------------------------------------------------------
    # Classification Head
    # --------------------------------------------------------------------------

    classifier_layers = [
        GlobalAveragePooling2D()
    ]


    if activation == "relu6":

        classifier_layers.extend(
            [
                Dense(
                    256,
                    activation=None,
                    kernel_regularizer=l2(1e-4)
                ),
                ReLU(max_value=6.0)
            ]
        )

    else:

        classifier_layers.append(
            Dense(
                256,
                activation=activation,
                kernel_regularizer=l2(1e-4)
            )
        )


    classifier_layers.extend(
        [
            Dropout(dropout_rate),
            Dense(1, activation="sigmoid")
        ]
    )


    model = Sequential(
        [
            base_model,
            *classifier_layers
        ]
    )


    # --------------------------------------------------------------------------
    # Model Compilation
    # --------------------------------------------------------------------------

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )


    # --------------------------------------------------------------------------
    # Training
    # --------------------------------------------------------------------------

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=0
    )


    history = model.fit(
        train_generator,
        validation_data=validation_generator,
        epochs=15,

        steps_per_epoch=len(train_generator),
        validation_steps=len(validation_generator),

        callbacks=[
            early_stopping
        ],

        verbose=0
    )


    # Return the highest validation accuracy achieved during training
    validation_accuracy = max(
        history.history["val_accuracy"]
    )

    return validation_accuracy

In [ ]:
# ==============================================================================
# Run Optuna Optimization
# ==============================================================================

study = optuna.create_study(
    direction="maximize"
)


study.optimize(
    objective,
    n_trials=30
)


print(
    "Best hyperparameters found by Optuna:"
)

print(
    study.best_trial.params
)


best_params = study.best_trial.params

In [ ]:
# ==============================================================================
# Final Selected Hyperparameters
# ==============================================================================
#
# These values were selected from the optimization process and are retained
# exactly for final model training.
#
# ==============================================================================

best_params = {
    "batch_size": 4,
    "activation": "relu6",
    "learning_rate": 0.00014907658309061954,
    "dropout_rate": 0.3554154155714014
}

In [ ]:
# ==============================================================================
# Final Model Training
# ==============================================================================
#
# Training is performed in two stages:
#
# Phase 1:
#   Train the custom classification head while keeping the pretrained
#   MobileNet feature extractor frozen.
#
# Phase 2:
#   Fine-tune the final MobileNet layers to adapt pretrained features
#   to crack classification.
#
# ==============================================================================


# ==============================================================================
# Data Generators
# ==============================================================================

train_generator = datagen.flow(
    X_train,
    y_train,
    batch_size=best_params["batch_size"],
    shuffle=True
)


validation_generator = datagen.flow(
    X_val,
    y_val,
    batch_size=best_params["batch_size"],
    shuffle=False
)


test_generator = datagen.flow(
    X_test,
    y_test,
    batch_size=best_params["batch_size"],
    shuffle=False
)



# ==============================================================================
# MobileNet Backbone
# ==============================================================================

base_model = MobileNet(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


# Freeze pretrained feature extractor for initial training
base_model.trainable = False



# ==============================================================================
# Classification Head
# ==============================================================================

classifier_layers = [
    GlobalAveragePooling2D()
]


if best_params["activation"] == "relu6":

    classifier_layers.extend(
        [
            Dense(
                256,
                activation=None,
                kernel_regularizer=l2(1e-5)
            ),

            ReLU(max_value=6.0)
        ]
    )

else:

    classifier_layers.append(
        Dense(
            256,
            activation=best_params["activation"],
            kernel_regularizer=l2(1e-5)
        )
    )


classifier_layers.extend(
    [
        Dropout(
            best_params["dropout_rate"]
        ),

        Dense(
            1,
            activation="sigmoid"
        )
    ]
)



model = Sequential(
    [
        base_model,
        *classifier_layers
    ]
)



# ==============================================================================
# Initial Model Compilation
# ==============================================================================

model.compile(
    optimizer=Adam(
        learning_rate=best_params["learning_rate"]
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]
)



# ==============================================================================
# Training Callbacks
# ==============================================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)


reduce_learning_rate = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=5,
    min_lr=1e-6,
    verbose=1
)



# ==============================================================================
# Phase 1: Train Classification Head
# ==============================================================================

history_phase1 = model.fit(
    train_generator,

    validation_data=validation_generator,

    epochs=10,

    steps_per_epoch=len(train_generator),
    validation_steps=len(validation_generator),

    callbacks=[
        early_stopping,
        reduce_learning_rate
    ],

    verbose=1
)



# ==============================================================================
# Phase 2: Fine-Tune MobileNet Layers
# ==============================================================================

# Enable MobileNet training
base_model.trainable = True


# Keep earlier convolutional layers frozen
for layer in base_model.layers[:-30]:
    layer.trainable = False



# Recompile with a lower learning rate
model.compile(
    optimizer=Adam(
        learning_rate=best_params["learning_rate"] * 0.1
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]
)



history_phase2 = model.fit(
    train_generator,

    validation_data=validation_generator,

    epochs=50,

    steps_per_epoch=len(train_generator),
    validation_steps=len(validation_generator),

    callbacks=[
        early_stopping,
        reduce_learning_rate
    ],

    verbose=1
)

In [ ]:
# ==============================================================================
# Combine Training Histories
# ==============================================================================

train_loss = (
    history_phase1.history["loss"] +
    history_phase2.history["loss"]
)


validation_loss = (
    history_phase1.history["val_loss"] +
    history_phase2.history["val_loss"]
)


train_accuracy = (
    history_phase1.history["accuracy"] +
    history_phase2.history["accuracy"]
)


validation_accuracy = (
    history_phase1.history["val_accuracy"] +
    history_phase2.history["val_accuracy"]
)


epochs = range(
    1,
    len(train_loss) + 1
)



# ==============================================================================
# Training Performance Visualization
# ==============================================================================

plt.figure(
    figsize=(11, 4)
)


# Loss curve
plt.subplot(
    1,
    2,
    1
)

plt.plot(
    epochs,
    train_loss,
    "b-",
    label="Training Loss"
)

plt.plot(
    epochs,
    validation_loss,
    "r--",
    label="Validation Loss"
)

plt.title(
    "Training vs Validation Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.legend()



# Accuracy curve
plt.subplot(
    1,
    2,
    2
)

plt.plot(
    epochs,
    train_accuracy,
    "b-",
    label="Training Accuracy"
)

plt.plot(
    epochs,
    validation_accuracy,
    "r--",
    label="Validation Accuracy"
)

plt.title(
    "Training vs Validation Accuracy"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Accuracy"
)

plt.legend()



plt.tight_layout()


plt.savefig(
    "loss_accuracy_curves.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# ==============================================================================
# Model Evaluation
# ==============================================================================
#
# Evaluate the trained TensorFlow model on the unseen test dataset using:
#   - Accuracy
#   - Precision
#   - Recall
#   - F1-score
#
# ==============================================================================


test_loss, test_accuracy = model.evaluate(
    test_generator
)


print(
    f"Final Test Accuracy: {test_accuracy:.4f}"
)

print(
    f"Final Test Loss: {test_loss:.4f}"
)



# Generate prediction probabilities
y_pred_probabilities = model.predict(
    test_generator
)


# Convert probabilities into binary predictions
y_pred = (
    y_pred_probabilities > 0.5
).astype(int).flatten()



# Match ground truth size
y_true = y_test[:len(y_pred)]



# ==============================================================================
# Classification Metrics
# ==============================================================================

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred
)

recall = recall_score(
    y_true,
    y_pred
)

f1 = f1_score(
    y_true,
    y_pred
)



print("\nGeneral Performance Metrics")
print("-" * 35)

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1-Score : {f1:.4f}"
)



# ==============================================================================
# Confusion Matrix
# ==============================================================================

confusion = confusion_matrix(
    y_true,
    y_pred
)


plt.figure(
    figsize=(5, 4)
)


sns.heatmap(
    confusion,
    annot=True,
    fmt="d",
    cmap="Blues",

    xticklabels=[
        "Mortar",
        "Brick"
    ],

    yticklabels=[
        "Mortar",
        "Brick"
    ]
)


plt.title(
    "Confusion Matrix"
)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "True Label"
)


plt.tight_layout()


plt.savefig(
    "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()



# ==============================================================================
# ROC Curve
# ==============================================================================

false_positive_rate, true_positive_rate, _ = roc_curve(
    y_true,
    y_pred_probabilities
)


roc_auc = auc(
    false_positive_rate,
    true_positive_rate
)



plt.figure(
    figsize=(6, 4)
)


plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=f"AUC = {roc_auc:.2f}",
    c="blue"
)


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    c="red"
)


plt.title(
    "ROC Curve"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)


plt.legend()

plt.grid(
    True,
    linewidth=0.2
)


plt.savefig(
    "roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# ==============================================================================
# Export TensorFlow SavedModel
# ==============================================================================
#
# TensorFlow Keras 3 uses model.export() for SavedModel generation.
#
# ==============================================================================


saved_model_path = (
    "/content/mobilenet_crack_classifier"
)


model.export(
    saved_model_path
)

In [ ]:
# ==============================================================================
# Convert SavedModel to ONNX
# ==============================================================================

import tf2onnx


loaded_model = tf.saved_model.load(
    saved_model_path
)


inference_function = loaded_model.signatures["serve"]



@tf.function(
    input_signature=[
        tf.TensorSpec(
            [
                None,
                224,
                224,
                3
            ],

            tf.float32,

            name="input"
        )
    ]
)

def serving_function(inputs):
    """
    Wrapper function providing a clean ONNX-compatible interface.
    """

    outputs = inference_function(inputs)


    # Handle SavedModel outputs returned as dictionaries
    if isinstance(outputs, dict):
        outputs = outputs[next(iter(outputs.keys()))]


    return outputs



onnx_path = (
    "/content/mobilenet_crack_classifier.onnx"
)



model_proto, _ = tf2onnx.convert.from_function(
    serving_function,

    input_signature=[
        tf.TensorSpec(
            [
                None,
                224,
                224,
                3
            ],

            tf.float32,

            name="input"
        )
    ],

    opset=13,

    output_path=onnx_path
)



print(
    "ONNX export successful:",
    onnx_path
)